### README

Identify NAGNAG disease-related variants annottated in literature.

*Output coordinates at base 1

**Ref: Hinzpeter, Alexandre, et al. "Alternative splicing at a NAGNAG acceptor site as a novel phenotype modifier." PLoS genetics 6.10 (2010): e1001153.

### Requirements

In [ ]:
# Install
"""
pandas==2.2.3
"""

In [ ]:
# Import libraries
import pandas as pd

### Constants

In [ ]:
# Input: Path to EXTRA variants table
EXTRA_PATH = '/PATH/TO/extra.csv' 


# Input: reference genome annotation
GTF_PATH = '/PATH/TO/Homo_sapiens.GRCh38.111.gtf.gz'


# Output: set directory
OUTPUT_DIR = '/PATH/TO/OUTPUT'

### Functions

In [ ]:
# EXTRACT INTRONS COORDINATES FROM GTF FILE

def extract_introns(gtf_df: pd.DataFrame) -> pd.DataFrame:

    # Filter only exon entries
    exon_df = gtf_df[gtf_df['Type'] == 'exon'].copy()

    #replace , by ; in Attributes column
    exon_df['Attributes'] = exon_df['Attributes'].str.replace(',',';')
    #replace : by = in Attributes column
    exon_df['Attributes'] = exon_df['Attributes'].str.replace(':','=')

    # Extract parent mRNA (transcript_id)
    exon_df['mRNA'] = (
        exon_df['Attributes']
        .str.split('transcript_id "', n=1, expand=True)[1]
        .str.split('"', n=1, expand=True)[0]
    )

    # Extract parent gene (gene_id)
    exon_df['GENEID'] = (
        exon_df['Attributes']
        .str.split('gene_id "', n=1, expand=True)[1]
        .str.split('"', n=1, expand=True)[0]
    )

    # Sort by gene, transcript, and genomic start
    exon_df = exon_df.sort_values(by=['GENEID', 'mRNA', 'Start'])

    # Assign coordinates of the downstream exon ("below")
    exon_df['Start_Below'] = exon_df.groupby('mRNA')['Start'].shift(-1)
    exon_df['End_Below'] = exon_df.groupby('mRNA')['End'].shift(-1)

    # Drop rows where downstream exon doesn't exist
    exon_df = exon_df.dropna(subset=['Start_Below', 'End_Below']).copy()

    # Calculate intron coordinates
    exon_df['Intron_Start'] = exon_df['End'] + 1
    exon_df['Intron_End'] = exon_df['Start_Below'] - 1

    # Keep relevant columns and remove duplicates
    intron_df = exon_df[['SeqID', 'Intron_Start', 'Intron_End', 'GENEID', 'STR']]
    # rename columns
    intron_df = intron_df.rename(columns={
        'SeqID': 'CHR',
        'Intron_Start': 'START',
        'Intron_End': 'END'
    })
    intron_df = intron_df.drop_duplicates()

    return intron_df

In [ ]:
# GET SHORT AND LONG INTRON COORDINATES BASED ON THE ANNOTATED INTRON POSITION AT 3' END

def get_nagnag_introns(
        extra_df: pd.DataFrame, intron_df: pd.DataFrame, intron_pos_col='Longest_Intron_Pos'
        ) -> pd.DataFrame:

    df = extra_df.copy()
    df[f'{intron_pos_col}_2'] = df.apply(lambda row: row[intron_pos_col] - 3 if row["STR"] == "+" else row[intron_pos_col] + 3, axis=1)

    # Split by strand
    plus_df = df[df['STR'] == '+']
    minus_df = df[df['STR'] == '-']

    # '+' strand merge (Pos == Start)
    merged_plus = pd.merge(
        plus_df,
        intron_df,
        left_on=[f'{intron_pos_col}', 'CHR', 'STR'],
        right_on=['END', 'CHR', 'STR'],
        how='left').rename(columns={
        'END': 'longIE',
        'START': 'longIS'
    })
    merged_plus['shortIS'] =  merged_plus['longIS']
    merged_plus['shortIE'] =  merged_plus['longIE'] - 3

    merged2_plus = pd.merge(
        plus_df,
        intron_df,
        left_on=[f'{intron_pos_col}_2', 'CHR', 'STR'],
        right_on=['END', 'CHR', 'STR'],
        how='left').rename(columns={
        'END': 'shortIE',
        'START': 'shortIS'
    })
    merged2_plus['longIS'] =  merged2_plus['shortIS'] 
    merged2_plus['longIE'] =  merged2_plus['shortIE'] + 3

    # '-' strand merge (Pos == End)
    merged_minus = pd.merge(
        minus_df,
        intron_df,
        left_on=[intron_pos_col, 'CHR', 'STR'],
        right_on=['START', 'CHR', 'STR'],
        how='left'
    ).rename(columns={
        'START': 'longIS',
        'END': 'longIE'
    })
    merged_minus['shortIS'] =  merged_minus['longIS'] + 3
    merged_minus['shortIE'] =  merged_minus['longIE'] 

    merged2_minus = pd.merge(
        minus_df,
        intron_df,
        left_on=[f'{intron_pos_col}_2', 'CHR', 'STR'],
        right_on=['START', 'CHR', 'STR'],
        how='left' 
    ).rename(columns={
        'START': 'shortIS',
        'END': 'shortIE'
    })
    merged2_minus['longIS'] =  merged2_minus['shortIS'] - 3
    merged2_minus['longIE'] =  merged2_minus['shortIE'] 

    # Combine both
    merged_df = pd.concat([merged_plus, merged2_plus, merged_minus, merged2_minus], ignore_index=True)
    # drop nan
    merged_df = merged_df.dropna(subset=['longIS', 'longIE', 'shortIS', 'shortIE'])
    # drop column intron_pos_col_2
    merged_df = merged_df.drop(columns=[f'{intron_pos_col}_2'])

    return merged_df

### Analysis

##### 1. Select NAGNAG AGAIN variants

In [ ]:
# load EXTRA variants tables into a single dataframe
extra_df = pd.read_csv(EXTRA_PATH)
# change chr col type
extra_df['CHR'] = extra_df['CHR'].astype(str)

extra_df

##### 2. Get intronic coordinates of the selected NAGNAG AGAIN variants

2.1 Extract all annotated introns coordinates

In [ ]:
# genome annotation
gtf_df = pd.read_table(GTF_PATH,
                       names = ['SeqID', 'Source', 'Type', 'Start', 'End', 'Score', 'STR', 'Phase', 'Attributes'],
                       comment='#',
                       low_memory=False) 


# get intron coordinates from gtf file
intron_df = extract_introns(gtf_df)
intron_df

2.2 Find longest intron position of the variants

In [ ]:
extra_df['Longest_Intron_Pos'] = extra_df['POS'] - extra_df['DIST_ACCEPTOR']
extra_df

2.3 Get intronic coordiantes of the variants

In [ ]:
# merge with intron dataframe
extra_df = get_nagnag_introns(extra_df, intron_df, intron_pos_col='Longest_Intron_Pos')


extra_df

##### 3. Filter and save variants coordinates

In [ ]:
# keep only relevant columns
extra_df = extra_df[['GENE', 'CHR', 'STR', 
         'shortIS', 'shortIE', 'longIS', 'longIE',
         'POS', 'REF', 'ALT']].copy()

# drop duplicates
extra_df = extra_df.drop_duplicates()

extra_df

In [ ]:
# save the merged dataframe to a csv file
extra_df.to_csv(f'{OUTPUT_DIR}/extra_variants.csv', index=False)